# Phase 3: Data Ingestion and Cleaning
This notebook connects to the MySQL database, extracts the integrated dataset, and prepares it for analysis.

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load environment variables (from parent dir since notebook runs in /notebooks)
env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
if not os.path.exists(env_path):
    env_path = os.path.join(os.getcwd(), '.env') # Fallback if run from root
load_dotenv(env_path)

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME = os.getenv('DB_NAME', 'airline_analytics')

encoded_pwd = quote_plus(DB_PASSWORD) if DB_PASSWORD else ""
auth = f"{DB_USER}:{encoded_pwd}" if encoded_pwd else f"{DB_USER}"
conn_str = f"mysql+pymysql://{auth}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)

### Extract Flights and Routes

In [3]:
query_flights = """
SELECT f.*, r.OriginAirport, r.DestAirport, r.DistanceMiles
FROM Flights f
JOIN Routes r ON f.RouteID = r.RouteID
WHERE f.FlightStatus = 'Completed'
"""
df_flights = pd.read_sql(query_flights, con=engine)
df_flights.head()

,FlightID,RouteID,DepartureDateTime,AircraftType,TotalCapacity,FlightStatus,FuelCost,CrewCost,AirportCost,OtherOperatingCost,OriginAirport,DestAirport,DistanceMiles
0,F000002,R002,2025-01-01 19:00:00,Boeing 737,160,Completed,12506.34,4817.67,2403.03,1061.25,LAX,JFK,2475
1,F000003,R003,2025-01-01 13:45:00,Airbus A320,180,Completed,10781.33,3509.80,2708.73,1198.14,ORD,SFO,1846
2,F000004,R003,2025-01-01 16:30:00,Boeing 737,160,Completed,9574.92,3789.28,1653.32,879.93,ORD,SFO,1846
3,F000005,R004,2025-01-01 17:30:00,Boeing 737,160,Completed,10619.49,3705.38,2959.67,878.53,SFO,ORD,1846
4,F000006,R004,2025-01-01 15:30:00,Boeing 737,160,Completed,10568.40,3524.32,1841.85,789.39,SFO,ORD,1846


### Extract Bookings and Customers

In [4]:
query_bookings = """
SELECT b.*, c.Age, c.Gender, c.LoyaltyTier, c.PassengerType
FROM Bookings b
JOIN Customers c ON b.CustomerID = c.CustomerID
WHERE b.IsCancelled = False
"""
df_bookings = pd.read_sql(query_bookings, con=engine)
df_bookings.head()

,BookingID,FlightID,CustomerID,BookingDateTime,CabinClass,TicketPrice,AncillaryRevenue,IsCancelled,Age,Gender,LoyaltyTier,PassengerType
0,B0000001,F000001,C16408,2024-10-26 05:42:00,Economy,357.84,105.0,0,42,M,Silver,Leisure
1,B0000002,F000001,C17829,2024-10-31 20:29:00,Economy,334.39,70.0,0,53,M,None,Leisure
2,B0000003,F000001,C09444,2024-11-22 05:28:00,Economy,341.51,35.0,0,41,F,None,Leisure
3,B0000004,F000001,C45316,2024-10-11 20:32:00,Economy,367.33,0.0,0,27,F,Gold,Leisure
4,B0000005,F000001,C47196,2024-12-24 14:52:00,Economy,501.89,35.0,0,30,M,None,Business


### Save Processed Data
Saving to pickle for fast loading in subsequent notebooks.

In [5]:
# Ensure processed directory exists
base_dir = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
processed_dir = os.path.join(base_dir, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

# Save for downstream notebooks
df_flights.to_pickle(os.path.join(processed_dir, 'flights_clean.pkl'))
df_bookings.to_pickle(os.path.join(processed_dir, 'bookings_clean.pkl'))
print("Data extracted and saved successfully.")

Data extracted and saved successfully.
